# Исследование надежности заемщиков


Во второй части проекта вы выполните шаги 3 и 4. Их вручную проверит ревьюер.
Чтобы вам не пришлось писать код заново для шагов 1 и 2, мы добавили авторские решения в ячейки с кодом. 



## Откройте таблицу и изучите общую информацию о данных

**Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную `data`. Путь к файлу:**

`/datasets/data.csv`

In [1]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

**Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.**

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Задание 3. Выведите основную информацию о датафрейме с помощью метода `info()`.**

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

**Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.**

In [4]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

**Задание 5. В двух столбцах есть пропущенные значения. Один из них — `days_employed`. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`. Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.**

In [5]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

**Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.**

In [6]:
data['days_employed'] = data['days_employed'].abs()

**Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.**

In [7]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть. Тем более этот столбец не понадобится вам для исследования.

**Задание 8. Выведите перечень уникальных значений столбца `children`.**

In [8]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

**Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.**

In [9]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

**Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.**

In [10]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

**Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждого типа занятости `income_type`.**

In [11]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

In [12]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

**Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.**

In [13]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

**Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру. Проверьте остальные столбцы.**

In [14]:
data['education'] = data['education'].str.lower()

**Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.**

In [15]:
data.duplicated().sum()

71

In [16]:
data = data.drop_duplicates()

### Категоризация данных

**Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:**

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.


**Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()` и метод `apply()`.**

In [17]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [18]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

**Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.**

In [19]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

**Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

**Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.**

**Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.**

In [20]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [21]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Шаг 3. Исследуйте данные и ответьте на вопросы

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

In [22]:
data[['children', 'debt']]# построим таблицу с нужными параметрами

,children,debt
0,1,0
1,1,0
2,0,0
3,3,0
4,0,0
...,...,...
21520,1,0
21521,0,0
21522,1,1
21523,3,1


In [23]:
print(data.pivot_table(index=['children', 'debt']))

               days_employed  dob_years  education_id  family_status_id  \
children debt                                                             
0        0      94723.692452  46.495855      0.825223          1.111759   
         1      69580.878269  43.000000      0.900282          1.260583   
1        0      23650.415176  38.558203      0.772915          0.796975   
         1      14031.814874  36.567568      0.927928          0.907658   
2        0       5456.742791  35.850915      0.771259          0.448332   
         1       7474.829032  34.989691      0.958763          0.479381   
3        0       7916.448744  36.313531      0.815182          0.389439   
         1      14936.586321  36.000000      0.925926          0.555556   
4        0      12972.051859  35.729730      0.756757          0.459459   
         1       1433.570815  39.000000      1.000000          1.000000   
5        0       1447.901899  38.777778      1.222222          0.222222   

                total_in

При группировке данных по количеству детей мы видим, что наличие и количество детей не влияют на наличие задолжности. Задолжности есть у людей с детьми и без детей.Единственное, что хорошо видно по информации, что у людей с 5 детьми отсутсвуют задолжности полностью. (Думаю, что отсутствие задолжности у людей с 5 детьми следствие хорошо выработанной отвественности за своё будущее.)

**Вывод:**  Из открытой части таблицы мы видим, что задолжности есть у людей вне зависимости от количества и наличия детей.Например: строка 3- у человека 3 детей и нет задолжности, либо строка 21523- 3 детей и есть задолжность.

In [24]:
print(data.pivot_table(index=['children', 'debt'],aggfunc='sum'))

               days_employed  dob_years  education_id  family_status_id  \
children debt                                                             
0        0      1.234060e+09     605748         10751             14484   
         1      7.396447e+07      45709           957              1340   
1        0      1.032104e+08     168268          3373              3478   
         1      6.230126e+06      16236           412               403   
2        0      1.013863e+07      66611          1433               833   
         1      1.450117e+06       6788           186                93   
3        0      2.398684e+06      11003           247               118   
         1      4.032878e+05        972            25                15   
4        0      4.799659e+05       1322            28                17   
         1      5.734283e+03        156             4                 4   
5        0      1.303112e+04        349            11                 2   

               total_inc

In [25]:
print(data.groupby('children')['debt'].sum().reset_index())

   children  debt
0         0  1063
1         1   444
2         2   194
3         3    27
4         4     4
5         5     0


При группировке данных по количеству детей мы видим, что наличие и количество детей не влияют на наличие задолжности. Задолжности есть у людей с детьми и без детей.Единственное, что хорошо видно по информации, что у людей с 5 детьми отсутсвуют задолжности полностью. (Думаю, что отсутствие задолжности у людей с 5 детьми следствие хорошо выработанной отвественности за своё будущее.)

При отсуствии детей самое большое количество задолжностей, подозреваю потому что люди думают только о себе и своей судьбе. При увеличении количества детей мы видим спад количества задолжностей со размерно количеству детей. Тем самым при наличие детей,люди более отвественны, потому что несут отвественность не только за себя.

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [26]:
data[['family_status', 'debt']]# построим таблицу с нужными параметрами

,family_status,debt
0,женат / замужем,0
1,женат / замужем,0
2,женат / замужем,0
3,женат / замужем,0
4,гражданский брак,0
...,...,...
21520,гражданский брак,0
21521,женат / замужем,0
21522,гражданский брак,1
21523,женат / замужем,1


**Вывод:** При сравнение столбцов семейное положение и  наличием задолжности также не прослеживается связи. 

In [27]:
print(data.groupby(['family_status', 'debt']).size().reset_index(name='count'))

           family_status  debt  count
0  Не женат / не замужем     0   2523
1  Не женат / не замужем     1    273
2              в разводе     0   1105
3              в разводе     1     84
4         вдовец / вдова     0    888
5         вдовец / вдова     1     63
6       гражданский брак     0   3749
7       гражданский брак     1    385
8        женат / замужем     0  11334
9        женат / замужем     1    927


При рассмотрение вопроса с точки зрения семейного положения мы видим также наличие и отсутствие задолжностей у обоих категорий. Но у людей, находящихся в гражданском либо обычном браке чаще отсуствует задолжность, чем у людей одиноких. Думаю, что это связанно с взаимовыручкой партнера. То есть если у человека есть партнер- в экстренном случае он выручит и получиться избежать задолжности.

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [28]:
print(data.groupby(['total_income', 'debt']).size().reset_index(name='count'))# построим таблицу с нужными параметрами

       total_income  debt  count
0             20667     1      1
1             21205     0      1
2             21367     0      1
3             21695     0      1
4             21895     0      1
...             ...   ...    ...
18616       1711309     0      1
18617       1715018     0      1
18618       1726276     0      1
18619       2200852     1      1
18620       2265604     0      1

[18621 rows x 3 columns]


Доходы и наличие задолжности. Из выше указанной таблицы мы видим, что задолжность есть у людей с уровнем дохода до 21000 и при 2200000.Пр уровне дохода до 21000р. выдача займа является небезопасной  по понятным причинам: данная сумма недостаточна для самообеспечения и выплаты займа по минимальному уровню дохода. Присутствие задолжности при отметке 2200852р. уже может иметь разные причины.

**Вывод:** При сравнении дохода с задолжностью взаимосвязи также не найдены. Например в последних 5 строках видно: человек с доходм 89672 имеет задолжность, а человек с доходом 82047- не имеет.

In [29]:
print(data.groupby(['total_income_category', 'debt']).size().reset_index(name='count'))

  total_income_category  debt  count
0                     A     0     23
1                     A     1      2
2                     B     0   4660
3                     B     1    354
4                     C     0  14568
5                     C     1   1353
6                     D     0    328
7                     D     1     21
8                     E     0     20
9                     E     1      2


При доходе от 50000р. до 200000 самое большое количество заявок как положительных, так и с задолжностью.С моей точки зрения,при таковом доходе человеку еще недостаточно денег для жизни без безкредитной жизни, но и появляется возможность улучшения комфорта жизни.

#### 3.4 Как разные цели кредита влияют на его возврат в срок?

In [30]:
data[['purpose', 'debt']]# построим таблицу с нужными параметрами

,purpose,debt
0,покупка жилья,0
1,приобретение автомобиля,0
2,покупка жилья,0
3,дополнительное образование,0
4,сыграть свадьбу,0
...,...,...
21520,операции с жильем,0
21521,сделка с автомобилем,0
21522,недвижимость,1
21523,на покупку своего автомобиля,1


**Вывод:** Цель кредита также не влият на наличие задолжности

In [31]:
print(data.groupby(['purpose', 'debt']).size().reset_index(name='count'))

                                   purpose  debt  count
0                               автомобили     0    432
1                               автомобили     1     44
2                               автомобиль     0    450
3                               автомобиль     1     41
4                       высшее образование     0    406
..                                     ...   ...    ...
71              строительство недвижимости     1     54
72  строительство собственной недвижимости     0    587
73  строительство собственной недвижимости     1     41
74                         сыграть свадьбу     0    702
75                         сыграть свадьбу     1     58

[76 rows x 3 columns]


In [32]:
print(data.groupby(['purpose_category', 'debt']).size().reset_index(name='count'))

           purpose_category  debt  count
0    операции с автомобилем     0   3879
1    операции с автомобилем     1    400
2  операции с недвижимостью     0   9971
3  операции с недвижимостью     1    780
4     получение образования     0   3619
5     получение образования     1    369
6        проведение свадьбы     0   2130
7        проведение свадьбы     1    183


При анализе наличия задолжности относительно цели мы видим такую картину: 
1. получение высшего образования задолжности отсутствуют полностью 
2. сыграть свадьбу- самый маленький процент имеющих задолжности
3. самый большой процент не выплат по цели- строительство собственной недвижимости.
4. по автомобилям- самое большое количество заявок и процент задолжности является средним, чуть больше чем по кредитам на свадьбы.

Самый маленький процент по невыплатам по категории операции с недвижимостью, чуть больше по проведение свадьбы, самый большой- получение образования. Считаю что задолжности по получение образования скорее всего связанны с тем, что человек переходит на новую должность либо получает первую специальность, соотвественно меняется уровень дохода и возможность выплаты.

#### 3.5 Приведите возможные причины появления пропусков в исходных данных.

*Ответ:* Человеческий фактор, возможно были ошибки во время копирования, считывания или записи данных, изменения формата файла

#### 3.6 Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

*Ответ:* потому некоторые значения сильно выделяются среди большинства. В данном случае медианное значение дает более объективную информацию, чем среднее значение.

### Шаг 4: общий вывод.

Размер доходов, семейное положение, количество детей и цель кредита не дают гарантии для повышения надежности заемщиков.

Главной целью исследования было: по известным параметрам выявить закономерности надежности заемщика. 
На этапах предобработки первоначально я привела данные в таблице в порядок, удалив  и заполнив пропуски. После обработки файла получилось выявить несколько закономерностей, дающих самый большой результат по надежности клиента. 
Портрет наиболее надежного заемщика: 1. Доход от 21000р. 2. Семейное положение - женат/замужем. 3. Цель кредита: получение образования либо свадьба.
Считаю важным добавить выводы по взаимосвязи целей кредита и наличия задолжности: 
1. получение высшего образования задолжности отсутствуют полностью 
2. сыграть свадьбу- самый маленький процент имеющих задолжности
3. самый большой процент невыплат по цели- строительство собственной недвижимости.
4. по автомобилям- самое большое количество заявок и процент задолжности является средним, чуть больше чем по кредитам на свадьбы. 
Вывод: на 100% гарантии выплаты кредита к сожалению не дает ни один параметр, кроме цели получение высшего образования. Считаю что по данной цели могут быть представленна не вся информация.
Рекомендации: увеличить порог доходов от 21000р, так как это увеличит процент выплат.    
            
        
        
            

Главной целью исследования было: по известным параметрам выявить закономерности надежности заемщика. На этапах предобработки первоначально я привела данные в таблице в порядок, удалив и заполнив пропуски. После обработки файла получилось выявить несколько закономерностей, дающих самый большой результат по надежности клиента. Портрет наиболее надежного заемщика: 1. Доход от 21000р. 2. Семейное положение - женат/замужем. 3. Цель кредита: операции с недвижимостью. Считаю важным добавить выводы по взаимосвязи целей кредита и наличия задолжности:
Самый маленький процент по невыплатам по категории операции с недвижимостью, чуть больше по проведение свадьбы, самый большой- получение образования. Считаю что задолжности по получение образования скорее всего связанны с тем, что человек переходит на новую должность либо получает первую специальность, соотвественно меняется уровень дохода и возможность выплаты.
Вывод: на 100% гарантии выплаты кредита к сожалению не дает ни один параметр. Рекомендации: увеличить порог доходов от 21000р, так как это увеличит процент выплат.